#Automated Incident Root Cause Analysis via Close-Code Classification
###Week 01 : Pre Processing & Clean Incident Data

### Load Dataset

Manually Load Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('/content/incident_event_log.csv')
df.head()

Or Load Via Kaggle using kaggle.json file for api key

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{\n  "username": "muzammilmohd1301",\n  "key": "KGAT_b504d4d64ad331c7f13b9347a42c8fcc"\n}'}

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d vipulshinde/incident-response-log

Dataset URL: https://www.kaggle.com/datasets/vipulshinde/incident-response-log
License(s): CC0-1.0
100% 2.07M/2.07M [00:01<00:00, 1.99MB/s]



In [ ]:
import zipfile

with zipfile.ZipFile('incident-response-log.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/incident_event_log.csv')
df.head()

,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,...,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,29/2/2016 04:40,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00


###Basic Inspection

In [ ]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119998 entries, 0 to 119997
Data columns (total 34 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   incident_state           119998 non-null  object
 1   active                   119998 non-null  bool  
 2   reassignment_count       119998 non-null  int64 
 3   reopen_count             119998 non-null  int64 
 4   made_sla                 119998 non-null  bool  
 5   caller_id                119998 non-null  object
 6   opened_by                119998 non-null  object
 7   opened_at                119998 non-null  object
 8   sys_created_by           119998 non-null  object
 9   sys_created_at           119998 non-null  object
 10  sys_updated_by           119998 non-null  object
 11  sys_updated_at           119998 non-null  object
 12  contact_type             119998 non-null  object
 13  location                 119998 non-null  object
 14  category            

,0
incident_state,0
active,0
reassignment_count,0
reopen_count,0
made_sla,0
caller_id,0
opened_by,0
opened_at,0
sys_created_by,0
sys_created_at,0


###Drop Useless Columns

Typical useless columns:

IDs (if not useful for prediction)\
Columns with >70% missing



In [ ]:
drop_cols = [
    'number',
    'sys_mod_count',
    'opened_by', 'sys_created_by', 'sys_updated_by',
    'assigned_to', 'resolved_by',
    'caller_id', 'problem_id', 'rfc', 'vendor', 'caused_by',
    'resolved_at', 'closed_at'
]

df = df.drop(columns=drop_cols, errors='ignore')

print("Columns after dropping:", df.columns)

Columns after dropping: Index(['incident_state', 'active', 'reassignment_count', 'reopen_count',
       'made_sla', 'opened_at', 'sys_created_at', 'sys_updated_at',
       'contact_type', 'location', 'category', 'subcategory', 'u_symptom',
       'cmdb_ci', 'impact', 'urgency', 'priority', 'assignment_group',
       'knowledge', 'u_priority_confirmation', 'notify', 'closed_code'],
      dtype='object')


###Handle Missing Values
| Column Type | Method              |
| ----------- | ------------------- |
| Categorical | Fill with "Unknown" |
| Numeric     | Fill with median    |


In [ ]:
cat_cols = df.select_dtypes(include='object').columns
num_cols = df.select_dtypes(exclude='object').columns

# Fill categorical
df[cat_cols] = df[cat_cols].fillna('Unknown')

# Fill numerical
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

###Clean Text Data

In [ ]:
date_cols = ['opened_at', 'sys_created_at', 'sys_updated_at']

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Extract features
if 'opened_at' in df.columns:
    df['opened_hour'] = df['opened_at'].dt.hour
    df['opened_day'] = df['opened_at'].dt.day
    df['opened_month'] = df['opened_at'].dt.month

# Drop original date columns
df = df.drop(columns=date_cols, errors='ignore')

/tmp/ipykernel_4755/744014007.py:5: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors='coerce')
/tmp/ipykernel_4755/744014007.py:5: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors='coerce')
/tmp/ipykernel_4755/744014007.py:5: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors='coerce')


In [ ]:
bool_cols = ['active', 'made_sla', 'knowledge', 'u_priority_confirmation']

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

###Remove Duplicates

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("Final Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())

Final Shape: (88873, 22)

Missing Values:
 incident_state             0
active                     0
reassignment_count         0
reopen_count               0
made_sla                   0
contact_type               0
location                   0
category                   0
subcategory                0
u_symptom                  0
cmdb_ci                    0
impact                     0
urgency                    0
priority                   0
assignment_group           0
knowledge                  0
u_priority_confirmation    0
notify                     0
closed_code                0
opened_hour                0
opened_day                 0
opened_month               0
dtype: int64


In [ ]:
print(df['closed_code'].value_counts())

closed_code
13    53936
14    14334
16     7610
15     3294
12     2845
1      2153
2      1144
3      1077
7       653
11      602
0       394
10      392
9       214
6       115
8        73
4        34
5         3
Name: count, dtype: int64


###Save Clean Dataset

In [ ]:
df.to_csv('/content/cleaned_incidents.csv', index=False)